# Pedagogical Significance of Constructing Neural Networks from First Principles Using NumPy

#  https://youtu.be/w8yWXqWQYmU?si=emF3p53eXfn_wTD8




## Abstract
This lab demonstrates the systematic construction of a feedforward neural network entirely from scratch using NumPy. The exercise transcends high-level frameworks, compelling learners to directly engage with the mathematical foundations of artificial intelligence—linear algebra, optimization, and nonlinear activation dynamics. By forgoing abstractions, the lab offers an invaluable didactic tool for developing both theoretical understanding and computational intuition.

## Key Importance
1. **Epistemological Clarity**  
   Strips away black-box automation and illuminates the mechanics of forward propagation, backpropagation, and gradient descent.  
   
2. **Cognitive Scaffolding**  
   Serves as a conceptual bridge between theoretical mathematics and applied deep learning, reinforcing neural computation at the matrix-operation level.  

3. **Skill Formation**  
   Strengthens fluency in vectorization, algorithmic thinking, and NumPy proficiency—skills foundational to research and industry practice.  

4. **Research Readiness**  
   Provides the intellectual foundation to extend into more complex architectures, regularization methods, and optimization algorithms.  

5. **Independent Mastery**  
   Cultivates confidence in building models without reliance on high-level libraries, fostering adaptability and innovation.  

---

**In essence:**  
This lab is not merely a coding exercise, but an academic exploration into the epistemic roots of machine learning—transforming abstract equations into executable logic and forming the cornerstone for advanced study in deep learning.


# Neural Network from Scratch (NumPy)


## 1. Problem Setup
- Task: Digit classification on MNIST dataset (28×28 grayscale images).
- Input: Each image has 784 pixels (flattened).
- Labels: Digits 0–9.

## 2. Data Preparation
- Loaded MNIST CSV using pandas.
- Converted to NumPy arrays for matrix math.
- Normalized pixel values to [0, 1].
- Split dataset into:
  - Training set
  - Dev/validation set (first 1000 samples).
- Transposed so each column = one example.

## 3. Network Architecture
- Input layer (a⁰) → 784 nodes (pixels).
- Hidden layer (a¹) → 10 nodes with ReLU activation.
- Output layer (a²) → 10 nodes with Softmax activation.
- Parameters:
  - W1, b1 → weights/bias for hidden layer.
  - W2, b2 → weights/bias for output layer.

## 4. Forward Propagation
- Linear step:
  - Z1 = W1·X + b1
  - Z2 = W2·A1 + b2
- Activation functions:
  - A1 = ReLU(Z1)
  - A2 = Softmax(Z2) → outputs class probabilities.

## 5. Backward Propagation
- Compute errors:
  - dZ2 = A2 - one_hot(Y)
  - dZ1 = W2ᵀ·dZ2 ⊙ ReLU’(Z1)
- Compute gradients:
  - dW2, db2 from output layer.
  - dW1, db1 from hidden layer.
- Used ReLU derivative (1 if z > 0 else 0).

## 6. Parameter Updates
- Gradient descent update rule:
  - W = W - α·dW
  - b = b - α·db
- Learning rate α is a hyperparameter.

## 7. Training Loop
- Initialize parameters randomly.
- For given iterations:
  - Forward propagation.
  - Backward propagation.
  - Update weights & biases.
- Print accuracy every 10 iterations.

## 8. Results
- Training accuracy ≈ 80–84%.
- Dev set accuracy ≈ 85–86%.
- Performance limited by:
  - Only 1 hidden layer.
  - Small number of units.
- Improvements possible with:
  - More layers.
  - Regularization.
  - Advanced optimizers (Momentum, RMSprop, Adam).

## 9. Predictions & Testing
- Tested with random digits:
  - Correctly classified most digits.
  - Occasionally misclassified tricky cases (e.g., rotated 3 as 5).
- Showed image + predicted label side by side.

## 10. Takeaways
- Built a working neural network from scratch.
- Reinforced understanding of:
  - Forward propagation.
  - Backpropagation.
  - Gradient descent.
- No TensorFlow / PyTorch / scikit-learn → only NumPy + math.


In [3]:
# -----------------------------
# 1. Import Libs & Load Digits dataset (from sklearn)
# -----------------------------
import numpy as np
from sklearn.datasets import load_digits

# Load dataset
digits = load_digits()
X = digits.data.T / 16.0   # normalize (values are 0–16 → scale to 0–1)
Y = digits.target
m = X.shape[1]             # number of examples
n_features = X.shape[0]    # 64 (8x8 images)

# Train/dev split
split = 1400  # first 1400 for training, rest for dev
X_train, Y_train = X[:, :split], Y[:split]
X_dev, Y_dev = X[:, split:], Y[split:]

# -----------------------------
# 2. Helper functions
# -----------------------------
def init_params():
    W1 = np.random.rand(10, n_features) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1, W2, b2

def ReLU(Z):
    return np.maximum(0, Z)

def ReLU_deriv(Z):
    return Z > 0

def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))  # stability
    return expZ / expZ.sum(axis=0, keepdims=True)

def one_hot(Y):
    one_hot_Y = np.zeros((Y.max() + 1, Y.size))
    one_hot_Y[Y, np.arange(Y.size)] = 1
    return one_hot_Y

# -----------------------------
# 3. Forward propagation
# -----------------------------
def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

# -----------------------------
# 4. Backward propagation
# -----------------------------
def backward_prop(Z1, A1, Z2, A2, W2, X, Y):
    m = X.shape[1]
    one_hot_Y = one_hot(Y)

    dZ2 = A2 - one_hot_Y
    dW2 = (1 / m) * dZ2.dot(A1.T)
    db2 = (1 / m) * np.sum(dZ2, axis=1, keepdims=True)

    dZ1 = W2.T.dot(dZ2) * ReLU_deriv(Z1)
    dW1 = (1 / m) * dZ1.dot(X.T)
    db1 = (1 / m) * np.sum(dZ1, axis=1, keepdims=True)

    return dW1, db1, dW2, db2

# -----------------------------
# 5. Parameter update
# -----------------------------
def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 -= alpha * dW1
    b1 -= alpha * db1
    W2 -= alpha * dW2
    b2 -= alpha * db2
    return W1, b1, W2, b2

# -----------------------------
# 6. Predictions & accuracy
# -----------------------------
def get_predictions(A2):
    return np.argmax(A2, axis=0)

def get_accuracy(predictions, Y):
    return np.mean(predictions == Y)

# -----------------------------
# 7. Training loop
# -----------------------------
def gradient_descent(X, Y, iterations, alpha):
    W1, b1, W2, b2 = init_params()

    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2,
                                       dW1, db1, dW2, db2, alpha)

        if i % 10 == 0:
            predictions = get_predictions(A2)
            acc = get_accuracy(predictions, Y)
            print(f"Iteration {i}: Training Accuracy = {acc:.4f}")

    return W1, b1, W2, b2

# -----------------------------
# 8. Run training
# -----------------------------
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, iterations=200, alpha=0.1)

# -----------------------------
# 9. Test on dev set
# -----------------------------
_, _, _, A2_dev = forward_prop(W1, b1, W2, b2, X_dev)
dev_preds = get_predictions(A2_dev)
print("Validation Accuracy:", get_accuracy(dev_preds, Y_dev))

Iteration 0: Training Accuracy = 0.0721
Iteration 10: Training Accuracy = 0.1100
Iteration 20: Training Accuracy = 0.1936
Iteration 30: Training Accuracy = 0.3250
Iteration 40: Training Accuracy = 0.3971
Iteration 50: Training Accuracy = 0.4500
Iteration 60: Training Accuracy = 0.5207
Iteration 70: Training Accuracy = 0.6164
Iteration 80: Training Accuracy = 0.6943
Iteration 90: Training Accuracy = 0.7657
Iteration 100: Training Accuracy = 0.8050
Iteration 110: Training Accuracy = 0.8300
Iteration 120: Training Accuracy = 0.8507
Iteration 130: Training Accuracy = 0.8629
Iteration 140: Training Accuracy = 0.8700
Iteration 150: Training Accuracy = 0.8736
Iteration 160: Training Accuracy = 0.8800
Iteration 170: Training Accuracy = 0.8871
Iteration 180: Training Accuracy = 0.8979
Iteration 190: Training Accuracy = 0.9036
Validation Accuracy: 0.818639798488665


# Importance of Building a Neural Network from Scratch (NumPy)

## 1. Understanding the Fundamentals
- Implementing a neural network without high-level libraries (TensorFlow, PyTorch, scikit-learn) forces you to **engage directly with the math**: linear algebra, activation functions, forward propagation, and backpropagation.
- This builds an **intuitive grasp** of how weights, biases, and gradients interact to improve predictions.

## 2. Demystifying the "Black Box"
- Deep learning frameworks often hide complexity behind a few lines of code.
- By coding every step manually, you **see how the system learns**, making neural networks less of a "magic trick" and more of a logical process.

## 3. Skill Development
- Strengthens proficiency with **NumPy**, the foundation of numerical computing in Python.
- Reinforces **matrix operations**, **vectorization**, and **efficiency** in scientific programming.

## 4. Foundation for Advanced Models
- Once the basics are mastered, extending to:
  - Deeper networks with multiple hidden layers.
  - Alternative activation functions.
  - Regularization techniques (dropout, L2).
  - Advanced optimizers (Momentum, Adam).
- This lab serves as the **launchpad for scaling up** to state-of-the-art models.

## 5. Critical Thinking in ML
- Training from scratch highlights challenges such as:
  - Overfitting vs. generalization.
  - Hyperparameter tuning (learning rate, iterations).
  - Bias-variance trade-offs.
- These are essential considerations when moving to real-world AI applications.

## 6. Confidence & Independence
- By proving you can build a neural network from the ground up:
  - You become less reliant on libraries.
  - Gain confidence to debug, customize, and innovate.
  - Build a **solid mental model** that makes learning advanced frameworks much easier.

---

**In short:**  
This lab bridges theory and practice, transforming abstract equations into working code. It lays the **conceptual and practical foundation** for any aspiring AI/ML practitioner.
